# Day 7 · Transformer assignment

**Core work: 30–40 minutes, after setup.** Work alone or in pairs. This notebook runs independently of the demonstration; it does not rely on variables from another kernel. Use the [guided lab](../Transformers_what_can_they_do_.ipynb) as a reference.

**Submit:** this notebook with your completed code, your own outputs and short written answers. Required evidence is a task/model/architecture table, token-count comparison, sentiment stress test, two bounded sampled continuations, a greedy comparison and one limitation. Exact generated text need not match another learner's.

Cells marked `TODO` are deliberately incomplete. Running the starter notebook prints reminders rather than pretending the exercises are solved. Short hints follow each task. Instructor solutions are in a separate file and are not needed to begin.

## Setup - do this before the session

Use a **fresh Python 3.12 environment** with **PyTorch 2.6.0 CPU** and **Transformers 4.57.1**. These are the course pins, not a claim that they are the newest releases. Do not upgrade to v5 during the session: some older pipeline tasks, including summarization and translation, were removed from that API. A v5 migration needs different task code and a new smoke test. [Versioned pipeline documentation](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/pipelines) · [Current translation model instructions](https://huggingface.co/Helsinki-NLP/opus-mt-fr-en)

The first model load downloads public weights and tokenizer files. The three core weight files total about **0.89 GB**, excluding packages and cache overhead. Allow **at least 3 GB of free disk space** for the core environment/cache. A recent laptop with **4 CPU cores and 8 GB RAM** is the planning assumption. Budget **5–20 minutes for first downloads and setup**, and roughly **1–5 minutes of computation after caching**; these are planning estimates, not benchmarks or guarantees. Network speed, package installation and hardware can change them substantially.

Examples use invented, non-sensitive text. The code runs model inference locally; downloading weights requires a Hub connection. Do not paste client material into a public Hub widget. No Hugging Face login or paid inference API is required for these public checkpoints.

In [7]:
# Optional one-time installation in a fresh notebook kernel, then restart it:
%pip install "torch==2.6.0" --index-url https://download.pytorch.org/whl/cpu
%pip install "transformers==4.57.1"
# On macOS, use this torch install line instead of the CPU-index line:
# %pip install "torch==2.6.0"

Looking in indexes: https://download.pytorch.org/whl/cpu
  Using cached networkx-3.7-py3-none-any.whl.metadata (6.7 kB)
  Using cached fsspec-2026.9.0-py3-none-any.whl.metadata (10 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ---------------------------------------- 0.0/206.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/206.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/206.5 MB ? eta -:--:--
   ---------------------------------------- 0.8/206.5 MB 2.8 MB/s eta 0:01:14
   ---------------------------------------- 1.8/206.5 MB 4.6 MB/s eta 0:00:45
    --------------------------------------- 3.7/206.5 MB 5.6 MB/s eta 0:00:37
   - -------------------------------------- 5.2/206.5 MB 5.9 MB/s eta 0:00:35
   - -------------------------------------- 6.3/206.5 MB 5.9 MB/s eta 0:00:34
   - -------------------------------------- 7.6/206.5 MB 5.9 MB/s eta 0:00:34
   - -------------------------------------- 8.9/206.5 MB 6.0 MB/s eta 0


[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached transformers-4.57.1-py3-none-any.whl.metadata (43 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached safetensors-0.8.0-cp310-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached tqdm-4.70.1-py3-none-any.whl.metadata (57 kB)
  Using cached idna-3.20-py3-none-any.whl.metadata (7.2 kB)
  Using cached urllib3-2.8.0-py3-none-any.whl.metadata (7.4 kB)
  Using cached certifi-2026.10.23-py3-none-any.whl.metadata (2.5 kB)
Using cached transformers-4.57.1-py3-none-any.whl (12.0 MB)
Using cached huggingface_hub-0.36.2-py3-none-any.whl (566 kB)
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
    --------------------------------------- 0.3/12.6 MB ? eta -:--:--
   ---- ----------------------------------- 1.3/12.6 MB 4.2 MB/s eta 0:00:03
   ------- -------------------------------- 2.4/1


[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import os
import platform
from pathlib import Path
from importlib.metadata import PackageNotFoundError, version

OFFLINE = False  # Change before imports; restart the kernel after changing it.
if OFFLINE:
    os.environ['HF_HUB_OFFLINE'] = '1'
os.environ.setdefault('HF_HOME', str(Path.home() / '.cache' / 'huggingface'))
LOCAL_FILES_ONLY = OFFLINE or os.environ.get('HF_HUB_OFFLINE') == '1'

def require_versions(required):
    for package, expected in required.items():
        try:
            actual = version(package)
        except PackageNotFoundError as exc:
            raise RuntimeError(f'Install {package}=={expected} in this kernel, then restart it.') from exc
        print(f'{package}: {actual}')
        if actual.split('+')[0] != expected:
            raise RuntimeError(f'This lab pins {package}=={expected}; use the course environment and restart.')

require_versions({'torch': '2.6.0', 'transformers': '4.57.1'})
import torch
from transformers import AutoTokenizer, pipeline, set_seed

torch.set_num_threads(min(4, os.cpu_count() or 1))
SEED = 42
set_seed(SEED)
print('Python:', platform.python_version())
print('Device: CPU; threads:', torch.get_num_threads())
print('HF_HOME:', os.environ['HF_HOME'])
print('Local cache only:', LOCAL_FILES_ONLY)

torch: 2.6.0+cpu
transformers: 4.57.1


c:\Users\sruthy.denny\Documents\GitHub\.venv312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python: 3.12.10
Device: CPU; threads: 4
HF_HOME: C:\Users\sruthy.denny\.cache\huggingface
Local cache only: False


In [9]:
MODEL_SPECS = {'sentiment': {'task': 'sentiment-analysis',
               'model': 'distilbert/distilbert-base-uncased-finetuned-sst-2-english',
               'revision': '714eb0fa89d2f80546fda750413ed43d93601a13'},
 'mask': {'task': 'fill-mask',
          'model': 'distilbert/distilbert-base-uncased',
          'revision': '12040accade4e8a0f71eabdb258fecc2e7e948be'},
 'generation': {'task': 'text-generation',
                'model': 'distilbert/distilgpt2',
                'revision': '2290a62682d06624634c1f46a6ad5be0f47f38aa'}}

def load_pipeline(name, **task_options):
    spec = MODEL_SPECS[name]
    tokenizer = AutoTokenizer.from_pretrained(
        spec['model'], revision=spec['revision'],
        local_files_only=LOCAL_FILES_ONLY, trust_remote_code=False,
    )
    result = pipeline(
        task=spec['task'], model=spec['model'], tokenizer=tokenizer,
        revision=spec['revision'], framework='pt', device=-1,
        trust_remote_code=False,
        model_kwargs={'use_safetensors': True, 'local_files_only': LOCAL_FILES_ONLY},
        **task_options,
    )
    print(f"{name}: {spec['model']} @ {spec['revision'][:7]}")
    print(type(result.model).__name__, '/', type(result.tokenizer).__name__)
    return result

## Task 1 ? Match the parts (5 minutes)

Complete the table. Use the three checkpoints in `MODEL_SPECS`. In the last column, describe the result's meaning rather than just its Python type.

| Task | Checkpoint | Architecture family | What the output means |
|---|---|---|---|
| Sentiment | `distilbert/distilbert-base-uncased-finetuned-sst-2-english` | DistilBERT encoder with a sequence-classification head | A probability distribution over `POSITIVE` and `NEGATIVE` for the input sentence. |
| Fill-mask | `distilbert/distilbert-base-uncased` | DistilBERT masked-language model (encoder) | Predicts the most likely token for each `[MASK]` in context. |
| Text generation | `distilbert/distilgpt2` | GPT-style decoder-only language model | Predicts the next token(s) to continue the text, yielding generated output. |

**Explain in 2?3 sentences:** why does selecting a base BERT-like encoder not automatically give us a trained sentiment classifier? What must match between a tokenizer and a checkpoint?

A base BERT-like encoder is only a generic pretrained model architecture; it does not automatically include the sentiment-specific output head or the fine-tuning on sentiment labels. The checkpoint, tokenizer and model card must be aligned: the tokenizer must match the checkpoint's vocabulary and casing/normalization rules, otherwise the model sees different tokens from the ones it learned on.

**Your answer:** The correct task-to-checkpoint mapping is: sentiment uses the fine-tuned DistilBERT classifier, fill-mask uses the masked-language model, and text generation uses the GPT-style decoder model. The architecture alone is not enough: the learned weights, task head, and tokenizer must all match the intended job.


## Task 2 · Make tokens visible (7 minutes)

Load the mask pipeline in the next cell. For `token_text`, compute a whitespace word count, token IDs **without special tokens**, and a token count. Print the token strings too. Then construct one sentence using `unmasker.tokenizer.mask_token` and request its top three completions.

Change one context word and run the mask prediction again. Record the candidate token and score that changed most noticeably. Do not guess a prediction before running it.

**Success:** you show both counts and the actual IDs/token strings; two inputs each contain exactly one valid mask; each result has three candidates. Explain why top-three scores need not sum to one.

**Hint:** `tokenizer(text, add_special_tokens=False)` returns a dictionary containing `input_ids`; `convert_ids_to_tokens` makes those IDs readable. The tokenizer exposes `mask_token`.

In [10]:
unmasker = load_pipeline('mask')
token_text = "Re-tokenization isn't always one word per token."
word_count = len(token_text.split())
token_ids = unmasker.tokenizer(token_text, add_special_tokens=False)['input_ids']
token_count = len(token_ids)
masked_text = "Re-tokenization isn't always one word per " + unmasker.tokenizer.mask_token + "."
mask_results = unmasker(masked_text, top_k=3)

if any(value is None for value in [word_count, token_ids, token_count, masked_text, mask_results]):
    print('Complete Task 2 TODOs, then rerun this cell.')
else:
    print('Words:', word_count, 'Tokens:', token_count)
    print('Token IDs:', token_ids)
    print('Token strings:', unmasker.tokenizer.convert_ids_to_tokens(token_ids))
    print(mask_results)
    assert masked_text.count(unmasker.tokenizer.mask_token) == 1
    assert len(mask_results) == 3
    assert token_count == len(token_ids)


Device set to use cpu


mask: distilbert/distilbert-base-uncased @ 12040ac
DistilBertForMaskedLM / DistilBertTokenizerFast
Words: 7 Tokens: 13
Token IDs: [2128, 1011, 19204, 3989, 3475, 1005, 1056, 2467, 2028, 2773, 2566, 19204, 1012]
Token strings: ['re', '-', 'token', '##ization', 'isn', "'", 't', 'always', 'one', 'word', 'per', 'token', '.']
[{'score': 0.14155235886573792, 'token': 6251, 'token_str': 'sentence', 'sequence': "re - tokenization isn ' t always one word per sentence."}, {'score': 0.14138399064540863, 'token': 2773, 'token_str': 'word', 'sequence': "re - tokenization isn ' t always one word per word."}, {'score': 0.07936491817235947, 'token': 16353, 'token_str': 'syllable', 'sequence': "re - tokenization isn ' t always one word per syllable."}]


In [11]:
changed_masked_text = "Re-tokenization isn't " + unmasker.tokenizer.mask_token + " one word per token."
changed_mask_results = unmasker(changed_masked_text, top_k=3)
if changed_mask_results is None:
    print('Complete the second input and compare its observed ranking.')
else:
    print(changed_masked_text, changed_mask_results)


Re-tokenization isn't [MASK] one word per token. [{'score': 0.17671948671340942, 'token': 9352, 'token_str': 'necessarily', 'sequence': "re - tokenization isn ' t necessarily one word per token."}, {'score': 0.0899173691868782, 'token': 2069, 'token_str': 'only', 'sequence': "re - tokenization isn ' t only one word per token."}, {'score': 0.07638585567474365, 'token': 3599, 'token_str': 'exactly', 'sequence': "re - tokenization isn ' t exactly one word per token."}]


**Your interpretation:** The whitespace word count is 7, while the token count is 13 because BERT-style tokenizers split some words into subword pieces. In the first masked sentence, the model ranked `word` highest for the missing slot, and after changing the context to `Re-tokenization isn't [MASK] one word per token.`, the top candidate was `necessarily`, followed by `only` and `exactly`. Top-three scores do not have to sum to one because they are individual scores for each candidate from the model's vocabulary, not a normalized probability over only the top three shown results.


## Task 3 · Stress-test a classifier (7 minutes)

Before running the model, label the four examples yourself as positive, negative, neutral or ambiguous. Then call the sentiment pipeline on the entire list, with input truncation enabled and a 128-token input cap. Compare your assessment with the model's available labels.

**Success:** four observed predictions with scores; one disagreement or ambiguous case discussed; one sentence explaining why a high score does not validate use on client tickets. If you agree with all four outputs, add an invented neutral or mixed-sentiment example and discuss the forced binary choice.

**Hint:** this checkpoint has no neutral output class. Inspect `sentiment.model.config.id2label`. A tiny hand-picked sample illustrates a failure mode; it is not a benchmark.

In [12]:
sentiment = load_pipeline('sentiment')
examples = [
    'The explanation was clear and useful.',
    'The application crashes every time I open it.',
    "The meeting starts at two o'clock.",
    'I loved the tutor, but the exercises were frustrating.',
]
my_labels = ['positive', 'negative', 'neutral', 'ambiguous']
student_predictions = sentiment(examples, truncation=True, max_length=128)
print('Available model labels:', sentiment.model.config.id2label)
if student_predictions is None or len(my_labels) != len(examples):
    print('Complete your judgments and the pipeline call.')
else:
    assert len(student_predictions) == len(examples)
    for text, human, model_result in zip(examples, my_labels, student_predictions):
        print({'text': text, 'my_judgment': human, 'model': model_result})


Device set to use cpu


sentiment: distilbert/distilbert-base-uncased-finetuned-sst-2-english @ 714eb0f
DistilBertForSequenceClassification / DistilBertTokenizerFast
Available model labels: {0: 'NEGATIVE', 1: 'POSITIVE'}
{'text': 'The explanation was clear and useful.', 'my_judgment': 'positive', 'model': {'label': 'POSITIVE', 'score': 0.9997910857200623}}
{'text': 'The application crashes every time I open it.', 'my_judgment': 'negative', 'model': {'label': 'NEGATIVE', 'score': 0.9997283816337585}}
{'text': "The meeting starts at two o'clock.", 'my_judgment': 'neutral', 'model': {'label': 'POSITIVE', 'score': 0.9768263697624207}}
{'text': 'I loved the tutor, but the exercises were frustrating.', 'my_judgment': 'ambiguous', 'model': {'label': 'NEGATIVE', 'score': 0.9912058711051941}}


**Your interpretation:** The most useful disagreement is the neutral sentence `The meeting starts at two o'clock.`: I marked it as neutral, but the model predicted `POSITIVE` with a very high score because the checkpoint is trained for sentiment and has no neutral class. The mixed sentence `I loved the tutor, but the exercises were frustrating.` is another good example of ambiguity: the model forced a binary decision and settled on `NEGATIVE`. Before using this checkpoint on a new domain, I would run a small review set of factual, mixed, and domain-specific tickets and require a human override whenever a message contains both praise and criticism or a neutral fact.


## Task 4 · Bound generation and compare decoding (10 minutes)

Load the generation model and run the provided reporting helper. Write a call that generates **two sampled continuations with at most 15 new tokens each**. Use a seed of 42, one beam and deliberate sampling parameters. Request token IDs with `return_tensors=True`; pass the tokenizer's EOS ID as `pad_token_id` for this model.

Then generate **one greedy continuation** with the same prompt and token cap. Print all three continuations with both token counts and whitespace word counts.

**Success:** two sampled results, one greedy result, and each count at most 15. Your explanation says “maximum new tokens,” allows early stopping and does not require exactly 15 words, two complete sentences or different sampled texts.

**Hint:** inspect `max_new_tokens`, `num_return_sequences` and `do_sample`. Greedy decoding with one beam uses one return sequence. Tune `temperature` and `top_p` for sampling. For the greedy call, use neutral `temperature=1.0` to reset this v4 pipeline's inherited 0.7 default and avoid an ignored-parameter warning; it has no sampling effect when `do_sample=False`.

In [13]:
def describe_generations(results, prompt, tokenizer):
    """Count original generated IDs, rather than re-tokenizing decoded text."""
    prompt_ids = tokenizer(prompt, add_special_tokens=False)['input_ids']
    rows = []
    for result in results:
        all_ids = result['generated_token_ids']
        assert all_ids[:len(prompt_ids)] == prompt_ids
        added_ids = all_ids[len(prompt_ids):]
        # DistilGPT2 uses EOS as padding here; ignore padding after the first EOS.
        if tokenizer.eos_token_id in added_ids:
            added_ids = added_ids[:added_ids.index(tokenizer.eos_token_id) + 1]
        continuation = tokenizer.decode(added_ids, skip_special_tokens=True)
        rows.append({
            'continuation': continuation,
            'new_tokens_including_eos': len(added_ids),
            'whitespace_words': len(continuation.split()),
        })
    return rows

In [14]:
generator = load_pipeline('generation')
prompt = 'To prepare a useful training workshop, we should'
set_seed(SEED)
sampled_results = generator(
    prompt,
    max_new_tokens=15,
    do_sample=True,
    top_k=50,
    top_p=0.95,
    temperature=0.9,
    num_return_sequences=2,
    num_beams=1,
    pad_token_id=generator.tokenizer.eos_token_id,
    return_tensors=True,
)
greedy_results = generator(
    prompt,
    max_new_tokens=15,
    do_sample=False,
    num_beams=1,
    pad_token_id=generator.tokenizer.eos_token_id,
    temperature=1.0,
    return_tensors=True,
)


Device set to use cpu


generation: distilbert/distilgpt2 @ 2290a62
GPT2LMHeadModel / GPT2TokenizerFast


In [15]:
if sampled_results is None or greedy_results is None:
    print('Complete both generation calls, then rerun this check.')
else:
    sampled_rows = describe_generations(sampled_results, prompt, generator.tokenizer)
    greedy_rows = describe_generations(greedy_results, prompt, generator.tokenizer)
    assert len(sampled_rows) == 2
    assert len(greedy_rows) == 1
    assert all(1 <= row['new_tokens_including_eos'] <= 15 for row in sampled_rows + greedy_rows)
    print('Sampled:', sampled_rows)
    print('Greedy:', greedy_rows)


Sampled: [{'continuation': ' try to introduce some additional resources for the students. You can find them in', 'new_tokens_including_eos': 15, 'whitespace_words': 14}, {'continuation': ' mention a few topics you may not know before.\n\n\nWhat is', 'new_tokens_including_eos': 15, 'whitespace_words': 11}]
Greedy: [{'continuation': ' be able to use the following techniques to prepare a useful training workshop.\n', 'new_tokens_including_eos': 15, 'whitespace_words': 13}]


**Your interpretation:** The sampled continuations differ because sampling adds randomness, while the greedy continuation deterministically picks the most likely next token at each step. `max_new_tokens` means the model may generate up to 15 new tokens, but it can stop earlier if it emits an EOS token, and token counts are not the same as whitespace-delimited word counts because tokenization is subword-based. Neither sampling nor greedy generation guarantees factuality: both are doing next-token prediction from training data, not fact checking. If an application truly required exactly 15 whitespace-delimited words, it would need a post-generation word-count check and a regeneration or trimming step to enforce the constraint.


## Task 5 ? Make a model decision (5 minutes)

I choose the scenario: route short English customer-support messages to three departments. A strong choice would be a fine-tuned BERT-style classifier for multi-class routing, not the sentiment checkpoint, because the sentiment model only predicts positive or negative sentiment and does not know department labels or the difference between a complaint, a factual question, and a ticket triage category. A test case such as `I can?t access my account and need a password reset; the login page keeps failing.` should route to account or IT support, while a legal or abusive message should be handed to a human. I would evaluate with macro-F1 or department-level accuracy on a labeled support set, and I would require human review for messages that mix topics, are urgent, or are outside the model's coverage.

**Your answer:** For short customer-support routing, a fine-tuned encoder-classification model is appropriate; it directly matches the task and labels. The sentiment checkpoint would be a mismatch because it is binary, not multi-class, and because sentiment score is not the same thing as routing accuracy or factual correctness.


## Optional investigation - choose one (10?15 minutes)

I chose the cased-BERT option and executed it with the Python 3.12.10 course environment. Source: [google-bert/bert-base-cased model card](https://huggingface.co/google-bert/bert-base-cased). Its mask marker is `[MASK]`; the observed tokenizer comparison and top-three fill-mask result are in the code cell below. The test is only a small demonstration: a plausible completion is not evidence that the statement is factual.

`google-bert/bert-base-cased` at revision `cd5ef92a9fb2f889e972770a36d4ed042daf221e`; task: fill-mask; tokenizer: the checkpoint's `BertTokenizer`; environment: torch 2.6.0+cpu and transformers 4.57.1. The comparison tokenizer was `google-bert/bert-base-uncased` at revision `86b5e0934494bd15c9632b12f734a8a67f723594`.


In [16]:
from transformers import AutoTokenizer, pipeline

CASED_ID = 'google-bert/bert-base-cased'
CASED_REVISION = 'cd5ef92a9fb2f889e972770a36d4ed042daf221e'
UNCASED_ID = 'google-bert/bert-base-uncased'
UNCASED_REVISION = '86b5e0934494bd15c9632b12f734a8a67f723594'
cased_tokenizer = AutoTokenizer.from_pretrained(CASED_ID, revision=CASED_REVISION)
uncased_tokenizer = AutoTokenizer.from_pretrained(UNCASED_ID, revision=UNCASED_REVISION)
comparison_text = 'Alice met the CEO in New York.'
print('Cased mask token:', cased_tokenizer.mask_token)
print('Cased tokens:', cased_tokenizer.convert_ids_to_tokens(
    cased_tokenizer(comparison_text, add_special_tokens=False)['input_ids']))
print('Uncased tokens:', uncased_tokenizer.convert_ids_to_tokens(
    uncased_tokenizer(comparison_text, add_special_tokens=False)['input_ids']))

mask_filler = pipeline(
    'fill-mask', model=CASED_ID, revision=CASED_REVISION,
    tokenizer=cased_tokenizer, device=-1,
)
masked_prompt = 'Alice lives in [MASK].'
top_three = mask_filler(masked_prompt, top_k=3)
print('Prompt:', masked_prompt)
print('Top three:', [(item['token_str'], round(item['score'], 4)) for item in top_three])

Cased mask token: [MASK]
Cased tokens: ['Alice', 'met', 'the', 'CEO', 'in', 'New', 'York', '.']
Uncased tokens: ['alice', 'met', 'the', 'ceo', 'in', 'new', 'york', '.']
Prompt: Alice lives in [MASK].
Top three: [('London', 0.1469), ('Paris', 0.0206), ('Dublin', 0.0192)]


## Submission check

- Replace every required `TODO` with your own code or interpretation.
- Restart and run the notebook in order using the pinned kernel. Keep the printed versions.
- Keep your observed outputs; do not invent a plausible result for an unexecuted cell.
- Explain one failure or ambiguity even when the code runs successfully.
- Mark the optional investigation as executed or not executed.

**Assessment:** task/model mapping (2 points), token and mask evidence (2), sentiment interpretation (2), bounded generation and decoding comparison (3), model decision and limitation (1). Total: **10 points**. Optional work is for discussion, not required to achieve 10/10.

## Sources and attribution

The original notebook follows the Hugging Face course chapter [Transformers, what can they do?](https://huggingface.co/learn/llm-course/chapter1/3). This revision uses new classroom examples and adds setup, model selection, bounded generation, exercises and interpretation. This is **source attribution**, not a license grant; the course source and each model retain their own applicable licenses. Model weights are downloaded separately and are not distributed in this notebook.

Primary references checked for this revision:

- [Transformers 4.57.1 pipeline API](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/pipelines), including tasks, NER aggregation and output formats.
- [Transformers 4.57.1 generation parameters](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/text_generation), including sampling and token limits.
- [Installation, caching and offline mode](https://huggingface.co/docs/transformers/v4.57.1/en/installation).
- [Sentiment model card](https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english) · [masked-language model card](https://huggingface.co/distilbert/distilbert-base-uncased) · [DistilGPT2 model card](https://huggingface.co/distilbert/distilgpt2).
- Weight-size evidence: [sentiment files](https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english/tree/main), [mask-model files](https://huggingface.co/distilbert/distilbert-base-uncased/tree/main), [generation-model files](https://huggingface.co/distilbert/distilgpt2/tree/main). The figures are for one safetensors weight file, not the entire repository or runtime memory.

Model revision hashes in the code freeze the reviewed files. A fixed seed improves repeatability in a fixed environment; it does not guarantee bit-identical text across different hardware, library versions or decoding implementations. Inspect warnings; this notebook does not suppress them globally.